# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to explore and process the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. We will:
- Load structured metadata from the Croissant schema,
- Inspect available record sets and fields by their `@id`,
- Extract tabular records for processing and EDA,
- Apply basic data cleaning and normalization,
- Visualize and summarize core characteristics of the dataset.

### Dataset Source
FAIR^2 dataset schema URL: https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Dataset (the Croissant schema)
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
metadata = dataset.metadata
print(metadata.name + ':')
print(metadata.description)

## 2. Data Overview
Explore available record sets (tables), their fields, and gather their `@id`s for precise referencing.

We will print the `@id` and the field information for each record set, as found in the dataset.

In [ ]:
# Discover all record sets available in the dataset, using their `@id`

record_set_dict = {}
if hasattr(metadata, 'record_sets'):
    for rs in metadata.record_sets:
        record_set_id = rs.id
        record_set_name = getattr(rs, 'name', '(no name)')
        print(f"Record Set @id: {record_set_id}")
        print(f"  Name: {record_set_name}")
        fields = getattr(rs, 'fields', [])
        record_set_dict[record_set_id] = [f.id for f in fields]
        print("  Fields:")
        for field in fields:
            print(f"    - @id: {field.id} (name: {getattr(field, 'name', '')}, type: {getattr(field, 'data_type', '')})")
        print()
else:
    # Try legacy key
    rs_list = getattr(metadata, 'record_set', [])
    for rs in rs_list:
        record_set_id = rs.id
        record_set_name = getattr(rs, 'name', '(no name)')
        print(f"Record Set @id: {record_set_id}")
        print(f"  Name: {record_set_name}")
        fields = getattr(rs, 'field', [])
        record_set_dict[record_set_id] = [f.id for f in fields]
        print("  Fields:")
        for field in fields:
            print(f"    - @id: {field.id} (name: {getattr(field, 'name', '')}, type: {getattr(field, 'data_type', '')})")
        print()

# Store all record set ids
record_set_ids = list(record_set_dict.keys())

**Example Rows**

Let's print a sample record from each discovered record set, using the record set's `@id`.

In [ ]:
# Print a sample of records per record set using their @id

for rs_id in record_set_ids:
    print(f"\nSample records from record set @id: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    for r in records[:2]:  # print only first 2 records for brevity
        print(r)

## 3. Data Extraction
Load entire data for each record set using its `@id` into a pandas DataFrame for further analysis.
- You can select a specific record set by its `@id` for detailed exploration.

In [ ]:
# Extract full data for each record set (@id).
# For demonstration, we focus on the main clinical data table. You can adjust `main_record_set_id` as needed.

dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Select the first record set for demonstration (modify as appropriate)
main_record_set_id = record_set_ids[0] if record_set_ids else None
print(f"\nColumns in record set '@id': {main_record_set_id}")
if main_record_set_id:
    print(list(dataframes[main_record_set_id].columns))
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

We demonstrate typical numeric filtering/normalization and group-by operations. All fields (columns) are referenced by their `@id` for future-proofing and reproducibility.

In [ ]:
# Example: Analyze a numeric field and a grouping field, referencing by @id.

df = dataframes[main_record_set_id]

# Choose numeric and group fields (edit these if you know the @id for your field of interest):
if len(df.columns) > 0:
    # Heuristically pick the first numeric-type column by trying float conversion
    for col in df.columns:
        try:
            # Try to convert to numeric and check if many values are valid numbers
            vec = pd.to_numeric(df[col], errors='coerce')
            if np.isfinite(vec).sum() > len(df)*0.3:
                numeric_field_id = col
                break
        except Exception:
            continue
else:
    numeric_field_id = None
# Similar for a categorical/grouping field
group_field_id = None
for col in df.columns:
    if col != numeric_field_id and df[col].nunique() < len(df)//2:
        group_field_id = col
        break

if numeric_field_id:
    # Cast column to numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    threshold = df[numeric_field_id].mean()  # example filter
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Group by categorical field (if present)
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame().reset_index()
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric fields found for EDA in this record set.")

## 5. Visualization

Visualize the distribution of the chosen numeric field and its relationship with the group field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    
    if group_field_id is not None:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion

- This notebook demonstrated how to load, inspect, and process real tabular biomedical records using `mlcroissant`. 
- All dataset components and columns were referenced using their Croissant `@id` for transparency and reproducibility.
- You may further tailor this analysis to focus on specific clinicopathological features (MSI status, anatomical site, comorbidities) by referencing their respective `@id`s from Section 2.
- FAIR^2, as a schema-driven package, supports robust sharing and downstream ML applications.